# Detección de errores

Este cuaderno busca detectar errores en los *raw data* y proponer soluciones. Los errores descritos en la memoria del proyecto son:
1. Como mínimo se tarda un día desde que la fruta se recoge hasta que llega al cliente final.
2. A cada proveedor no se le pueden vender en un mismo día más de 100kg de una misma fruta.
3. No hay marcas que produzcan más de un tipo de fruta.
4. Un mismo lote no puede tener marcas o frutas distintas

In [2]:
#importamos librerias
import pandas as pd
import numpy as np


#Cargamos el df
df_final=pd.read_csv("../df_final.csv", sep=",")
df_final

,id_unico,t_id,lote,tiempo_recogida,coste_inicial,proveedor,tipo,marca,cliente,tiempo_venta,precio_venta,peso
0,1,scene15781.png,C67K78K49Q55T49J80T71,322,2.021043,Ganadería Orgánica TierraFértil,Guava,DulceEncanto,Cosecha Fresca,330.0,3.068684,322.285621
1,2,scene15741.png,C67K78K49Q55P49J80T71,337,1.693417,EcoFungicidas Morales,Guava,MaravillaJugosa,CompraRápida,345.0,2.521239,230.210207
2,3,scene15721.png,C67K78K49Q55N49J80T71,642,1.848432,Reforestación Ecológica VerdeVida,Guava,AventuraFrutal,EcoMercado Sostenible,650.0,4.431501,189.927487
3,4,scene15681.png,C67K78K49Q54T49J80T71,407,3.432856,Tecnosembradoras del Sur,Guava,FrutaDulce,SuperValle Verde,414.0,4.801658,189.375125
4,5,scene15661.png,C67K78K49Q54R49J80T71,174,0.971298,Agricultura Inteligente TechCultivos,Guava,PaladarDorado,Comercial Fresco,174.0,2.745247,265.053854
...,...,...,...,...,...,...,...,...,...,...,...,...
70544,70545,109red applee01021109.png,M48U82K68G80V76K69L49L50M49L57J80T71,518,2.051245,Cosechadoras Progresivas SA,Apple,DulzuraSilvestre,SuperValle Verde,522.0,3.688873,265.530610
70545,70546,108red applee01006108.png,M48T82K68G80V76K69L49L48R49L56J80T71,412,2.147653,Fitosanitarios BioPro,Apple,FrutaDulce,EcoTienda,419.0,3.742457,189.384444
70546,70547,107red applee01001107.png,M48S82K68G80V76K69L49L48M49L55J80T71,391,2.787196,Silos y Almacenes AgroVault,Apple,TesoroNaturaleza,Distribuidora Nacional,403.0,3.035907,307.542502
70547,70548,103red applee00916103.png,M48O82K68G80V76K69L48U49R49L51J80T71,683,1.925670,Agronutrientes del Futuro,Apple,AventuraFrutal,La Canasta Feliz,683.0,3.477034,229.292057


In [4]:
df_final_copia=df_final.copy()

## Tiempo de recogida
Para poder proceder con este estudio de errores es necesario transformar los campos de tiempo_recogida y tiempo_venta a tipo date.

Posteriormente se calculará la diferencia entre ambos que deberá ser mayor a 24 h. Otra aproximación es calcular la diferencia entre los valores de horas.

In [5]:
print(list(df_final_copia.columns))

['id_unico', 't_id', 'lote', 'tiempo_recogida', 'coste_inicial', 'proveedor', 'tipo', 'marca', 'cliente', 'tiempo_venta', 'precio_venta', 'peso']


In [15]:
#Calculo la diferencia de las horas y lo guardo en una variable 
diferencia_tiempo_venta=df_final_copia["tiempo_venta"]- df_final_copia["tiempo_recogida"]
diferencia_tiempo_venta

#Posicionamos esta serie dentro del df_final
df_final_copia["diferencia_tiempo_venta"]=diferencia_tiempo_venta

#Vemos los clientes, lotes, marca, y tipo de fruta que cumplen y no cumplen el valor de 24h

lista_variables_estudio=["cliente","lote","marca","tipo"]

print("Casos de studios donde se cunple el tiempo de espera")

for variable in lista_variables_estudio:
    df_filtrado=df_final_copia[[variable, "diferencia_tiempo_venta"]][df_final_copia["diferencia_tiempo_venta"]>24]
    print(df_filtrado.value_counts())

Casos de studios donde se cunple el tiempo de espera
cliente                 diferencia_tiempo_venta
La Tienda Justa         105.0                      2
Distribuidora Alfa      188.0                      2
Distribuidora Nacional  352.0                      2
Alimentación Total      109.0                      1
MaxiAlimentos           416.0                      1
                                                  ..
Distribuciones del Sol  352.0                      1
                        344.0                      1
                        329.0                      1
                        305.0                      1
Tienda Selecta          550.0                      1
Name: count, Length: 402, dtype: int64
lote                   diferencia_tiempo_venta
C67K78K48L48L49J80T71  76.0                       1
C67K78K48R54L49J80T71  77.0                       1
C67K78K48R57L49J80T71  216.0                      1
C67K78K48R56R49J80T71  189.0                      1
C67K78K48R56N49J80T71 

In [16]:
for variable in lista_variables_estudio:
    df_filtrado=df_final_copia[[variable, "diferencia_tiempo_venta"]][df_final_copia["diferencia_tiempo_venta"]<24]
    print(df_filtrado.value_counts())

cliente                diferencia_tiempo_venta
Tienda Selecta          7.0                       314
Mercado del Barrio      6.0                       312
SuperValle Verde        7.0                       312
Distribuidora Alfa      7.0                       301
SuperMercado Ideal      7.0                       300
                                                 ... 
EcoTienda              -688.0                       1
EcoMercado Sostenible   17.0                        1
                       -14.0                        1
                       -25.0                        1
Tienda Selecta          17.0                        1
Name: count, Length: 1591, dtype: int64
lote                                  diferencia_tiempo_venta
G80V76K69L49N50J80T71                 7.0                        2
C67K78K48T52T49J80T71                 6.0                        2
C67K78K48L56T49J80T71                 7.0                        2
C67K78K48R57T49J80T71                 9.0               